In [1]:
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.metrics import (
    classification_report, accuracy_score, confusion_matrix, 
    f1_score, fbeta_score, 
    matthews_corrcoef, brier_score_loss, RocCurveDisplay
)
from sklearn.datasets import make_classification
from sklearn.svm import SVC
from sklearn.preprocessing import OneHotEncoder,MaxAbsScaler,FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.calibration import CalibrationDisplay

from imblearn.over_sampling import RandomOverSampler
from xgboost import XGBClassifier
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE
from sklearn.inspection import permutation_importance, partial_dependence


Goal: We want to maximize amount of money saved by Big G. A full derate costs around \\$4000 for towing, plus the cost of
repairs. The cost of a false positive prediction is about \\$500 due to having the truck oﬀ the road and serviced unnecessarily. Thus, we'll want to optimize our model for recall.

We want to attempt to predict an idle derate at least **2 hours** before it occurs.

In [2]:
faults = pd.read_csv("../data/J1939Faults.csv")

# Dropping columns that are not relevant
faults_clean = faults.drop(columns=['ecuSoftwareVersion', 'ecuSerialNumber', 'ecuModel', 'ecuMake', 'ecuSource', 'MCTNumber', 'ESS_Id', 'actionDescription', 'faultValue'])

faults_clean.head()

/var/folders/51/zgq0lbb14t13h0_bgn8nntnm0000gn/T/ipykernel_91283/4289917023.py:1: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  faults = pd.read_csv("../data/J1939Faults.csv")


,RecordID,EventTimeStamp,eventDescription,spn,fmi,active,activeTransitionCount,EquipmentID,Latitude,Longitude,LocationTimeStamp
0,1,2015-02-21 10:47:13.000,Low (Severity Low) Engine Coolant Level,111,17,True,2,1439,38.857638,-84.626851,2015-02-21 11:34:25.000
1,2,2015-02-21 11:34:34.000,NaN,629,12,True,127,1439,38.857638,-84.626851,2015-02-21 11:35:10.000
2,3,2015-02-21 11:35:31.000,Incorrect Data Steering Wheel Angle,1807,2,False,127,1369,41.421250,-87.767361,2015-02-21 11:35:26.000
3,4,2015-02-21 11:35:33.000,Incorrect Data Steering Wheel Angle,1807,2,True,127,1369,41.421018,-87.767361,2015-02-21 11:36:08.000
4,5,2015-02-21 11:39:41.000,NaN,4364,17,False,2,1674,38.416481,-89.442638,2015-02-21 11:39:37.000


Looking at the first record, here is a breakdown of the important values.

* ESS_Id, actionDescription, ecuSoftwareVersion, ecuSerialNumber, ecuModel, ecuMake, ecuSource, faultValue, and MCTNumber are unlikely to provide any predictive value.
* We can see the time of the event in the **EventTimeStamp** column. Note that this may be different from the **LocationTimeStamp** value, which indicates when the Latitude/Longitude values were recorded.
* The **spn** and **fmi** columns together indicate the type of fault, and there may be a description of that fault in the **eventDescription** column, although this column is sometimes missing.
* Faults are recorded when the light goes on and when it goes off, which is indicated by the **active** column, with True indicating the light turning on and False indicating turning off. The number of times the code has been set or unset is in the **faultValue** column, although this value can be unreliable. 
* Each truck has an identifier, the **EquipmentID** value.
* Each record can be linked to the on-board diagnostics data through the **RecordID** column.

Filtering out vehicles that are likely being serviced: (We'll come back to this)

Service stations are at (36.0666667, -86.4347222), (35.5883333, -86.4438888), and (36.1950, -83.174722)

In [3]:
# Creating a dataframe with service station locations
stations = pd.DataFrame({'lat':[36.0666667, 35.5883333, 36.1950], 'lon':[-86.4347222, -86.4438888, -83.174722]})

In [4]:
# From this article: https://kanoki.org/2019/12/27/how-to-calculate-distance-in-python-and-pandas-using-scipy-spatial-and-distance-functions/
# Vectorized Haversine formula
def haversine_vectorize(lon1, lat1, lon2, lat2):

    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    newlon = lon2 - lon1
    newlat = lat2 - lat1

    haver_formula = np.sin(newlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(newlon/2.0)**2

    dist = 2 * np.arcsin(np.sqrt(haver_formula ))
    mi = 3958 * dist 
    return mi

faults_clean['dist_from_stat_1'] = haversine_vectorize(stations.loc[0]['lon'], stations.loc[0]['lat'], faults_clean['Longitude'], faults_clean['Latitude'])
faults_clean['dist_from_stat_2'] = haversine_vectorize(stations.loc[1]['lon'], stations.loc[1]['lat'], faults_clean['Longitude'], faults_clean['Latitude'])
faults_clean['dist_from_stat_3'] = haversine_vectorize(stations.loc[2]['lon'], stations.loc[2]['lat'], faults_clean['Longitude'], faults_clean['Latitude'])


In [5]:
faults_clean.head()

,RecordID,EventTimeStamp,eventDescription,spn,fmi,active,activeTransitionCount,EquipmentID,Latitude,Longitude,LocationTimeStamp,dist_from_stat_1,dist_from_stat_2,dist_from_stat_3
0,1,2015-02-21 10:47:13.000,Low (Severity Low) Engine Coolant Level,111,17,True,2,1439,38.857638,-84.626851,2015-02-21 11:34:25.000,216.779342,246.957471,200.394786
1,2,2015-02-21 11:34:34.000,NaN,629,12,True,127,1439,38.857638,-84.626851,2015-02-21 11:35:10.000,216.779342,246.957471,200.394786
2,3,2015-02-21 11:35:31.000,Incorrect Data Steering Wheel Angle,1807,2,False,127,1369,41.421250,-87.767361,2015-02-21 11:35:26.000,376.784933,409.225406,437.407343
3,4,2015-02-21 11:35:33.000,Incorrect Data Steering Wheel Angle,1807,2,True,127,1369,41.421018,-87.767361,2015-02-21 11:36:08.000,376.769223,409.209647,437.394356
4,5,2015-02-21 11:39:41.000,NaN,4364,17,False,2,1674,38.416481,-89.442638,2015-02-21 11:39:37.000,231.732035,255.969764,376.935409


In [6]:
faults_clean.shape

(1187335, 14)

In [7]:
# Only keep rows that are more than 1 mile away from all 3 of the service stations
faults_clean = faults_clean[(faults_clean['dist_from_stat_1'] > 1) & (faults_clean['dist_from_stat_2'] > 1) & (faults_clean['dist_from_stat_3'] > 1)]


In [8]:
# Drop intermediate columns that we don't need
faults_clean = faults_clean.drop(columns=['dist_from_stat_1','dist_from_stat_2','dist_from_stat_3'])

In [9]:
faults_clean.shape

(1055071, 11)

In [10]:
(1187335-1055071)/1187335

0.11139568866410912

Looks like about 11% of our dataset was collected within 1 mile of a service station.

In [11]:
diagnostics = pd.read_csv("../data/VehicleDiagnosticOnboardData.csv")

diagnostics.head()

,Id,Name,Value,FaultId
0,1,IgnStatus,False,1
1,2,EngineOilPressure,0,1
2,3,EngineOilTemperature,96.74375,1
3,4,TurboBoostPressure,0,1
4,5,EngineLoad,11,1


Pivoting diagnostic onboard data by fault ID so that it's easier to merge. 

In [12]:
diagnostics[diagnostics['FaultId']==2]

,Id,Name,Value,FaultId
21,22,IgnStatus,True,2
22,23,LampStatus,1279,2


In [13]:
diagnostics_pivoted = diagnostics.pivot(index='FaultId', columns='Name', values='Value')

In [14]:
diagnostics_pivoted.head()

Name,AcceleratorPedal,BarometricPressure,CruiseControlActive,CruiseControlSetSpeed,DistanceLtd,EngineCoolantTemperature,EngineLoad,EngineOilPressure,EngineOilTemperature,EngineRpm,...,FuelTemperature,IgnStatus,IntakeManifoldTemperature,LampStatus,ParkingBrake,ServiceDistance,Speed,SwitchedBatteryVoltage,Throttle,TurboBoostPressure
FaultId,,,,,,,,,,,,,,,,,,,,,
1,0,14.21,False,66.48672,423178.7,100.4,11,0,96.74375,0,...,NaN,False,78.8,1023,True,NaN,0,3276.75,NaN,0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,16639,NaN,NaN,NaN,NaN,NaN,NaN


To merge diagnostic info with fault code info, we'll match up the **RecordID** and **FaultId**.

In [15]:
faults_full = pd.merge(faults_clean, diagnostics_pivoted, left_on='RecordID', right_on='FaultId', how='outer')

faults_full.head()

,RecordID,EventTimeStamp,eventDescription,spn,fmi,active,activeTransitionCount,EquipmentID,Latitude,Longitude,...,FuelTemperature,IgnStatus,IntakeManifoldTemperature,LampStatus,ParkingBrake,ServiceDistance,Speed,SwitchedBatteryVoltage,Throttle,TurboBoostPressure
0,1.0,2015-02-21 10:47:13.000,Low (Severity Low) Engine Coolant Level,111.0,17.0,True,2.0,1439,38.857638,-84.626851,...,NaN,False,78.8,1023,True,NaN,0,3276.75,NaN,0
1,2.0,2015-02-21 11:34:34.000,NaN,629.0,12.0,True,127.0,1439,38.857638,-84.626851,...,NaN,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
2,3.0,2015-02-21 11:35:31.000,Incorrect Data Steering Wheel Angle,1807.0,2.0,False,127.0,1369,41.421250,-87.767361,...,NaN,NaN,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
3,4.0,2015-02-21 11:35:33.000,Incorrect Data Steering Wheel Angle,1807.0,2.0,True,127.0,1369,41.421018,-87.767361,...,NaN,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
4,5.0,2015-02-21 11:39:41.000,NaN,4364.0,17.0,False,2.0,1674,38.416481,-89.442638,...,NaN,NaN,NaN,16639,NaN,NaN,NaN,NaN,NaN,NaN


Creating target column:

Our plan will be to make 4 different target columns: 
* Derate in 0-2 hours
* Derate in 2-6 hours
* Derate in 6-12 hours
* Derate in 12-24 hours

Then, we'll fit our model on each of the 4 target columns and see which model is most predictive.

The idea is to give the vehicle operator enough warning so that they have enough time to drive to a service station. 0-2 hours of warning probably isn't enough time. As we increase the window larger and larger, we'll probably lose some predictive power. So we'll try to find the sweet spot by evaluating several different time windows.

In [16]:
# Creating a new column to mark whether the fault represents a full derate
# We'll use this when we apply .rolling later 
faults_full['derate_flag'] = ((faults_full["spn"] == 5246) & (faults_full["active"] == True)).astype('int')

In [17]:
# Converting EventTimeStamp to datetime 
faults_full['EventTimeStamp']= pd.to_datetime(faults_full['EventTimeStamp'])

In [18]:
# Running into an issue with duplicated EquipmentID/EventTimeStamp. 
# The .rolling method can't handle the duplicated EquipmentID and EventTimeStamp columns.
# We'll need to de-duplicate before we apply .rolling
# By taking the max of 'derate_flag', we're ensuring that for each EquipmentID/EventTimeStamp combination, 
#       we're capturing whether at least one derate occurred for that vehicle at that time (if multiple faults occurred simultaneously)
faults_dedup = faults_full.groupby(['EquipmentID', 'EventTimeStamp']).max('derate_flag').reset_index()

In [19]:
faults_dedup.head()

,EquipmentID,EventTimeStamp,RecordID,spn,fmi,activeTransitionCount,Latitude,Longitude,derate_flag
0,301,2015-05-11 13:11:20,49415.0,639.0,2.0,127.0,36.189398,-82.795601,0
1,301,2015-05-13 08:22:32,51363.0,596.0,31.0,3.0,35.872500,-84.475648,0
2,301,2015-05-18 09:34:05,57330.0,3226.0,10.0,6.0,35.972870,-83.920555,0
3,301,2015-05-21 13:57:35,61706.0,639.0,2.0,127.0,36.384953,-86.478379,0
4,301,2015-05-21 14:54:32,61801.0,639.0,2.0,127.0,36.384814,-86.478379,0


In [20]:
# Ensure faults data is sorted by EventTimeStamp
# By sorting the EventTimeStamp in descending order, we're ensuring that we are rolling forwards in time (since .rolling looks at preceding rows)
faults_dedup = faults_dedup.sort_values(by='EventTimeStamp', ascending=False)

# Then we're applying the .rolling method with a certain time window and taking the rolling count of derates.
# The new columns are indicating the total number of derates that happened in the next certain time window for the given vehicle.
target = faults_dedup[['EquipmentID','EventTimeStamp']].copy()
                       
target['total_derates_2_hr'] = faults_dedup.groupby('EquipmentID').rolling(window='2h', on='EventTimeStamp')['derate_flag'].sum().reset_index(drop=True)
target['total_derates_6_hr'] = faults_dedup.groupby('EquipmentID').rolling(window='6h', on='EventTimeStamp')['derate_flag'].sum().reset_index(drop=True)
target['total_derates_12_hr'] = faults_dedup.groupby('EquipmentID').rolling(window='12h', on='EventTimeStamp')['derate_flag'].sum().reset_index(drop=True)
target['total_derates_24_hr'] = faults_dedup.groupby('EquipmentID').rolling(window='24h', on='EventTimeStamp')['derate_flag'].sum().reset_index(drop=True)

# Next we take successive differences so that we can get total number of derates within the time ranges we're interested in (2-6 hrs, 6-12 hrs, 12-24 hrs)
# If there were 1 or more derates in the given time range, we mark it with "1"; otherwise, "0"
target['derate_2_6_hr'] = np.where(target['total_derates_6_hr'] - target['total_derates_2_hr'] > 0, 1, 0)
target['derate_6_12_hr'] = np.where(target['total_derates_12_hr'] - target['total_derates_6_hr'] > 0, 1, 0)
target['derate_12_24_hr'] = np.where(target['total_derates_24_hr'] - target['total_derates_12_hr'] > 0, 1, 0)

In [21]:
# Drop intermediate columns that we don't need
target = target.drop(columns=['total_derates_2_hr','total_derates_6_hr','total_derates_12_hr','total_derates_24_hr'])

In [22]:
# Next, we'll merge the new target columns into the original DataFrame, merging on EventTimeStamp and EquipmentID
# Any duplicate EventTimeStamp/EquipmentID combinations will have the same value in the target column
faults_full = pd.merge(target, faults_full, on=['EventTimeStamp','EquipmentID'], how='inner')

# Ensure faults_full is sorted in order of EventTimeStamp
faults_full = faults_full.sort_values(by='EventTimeStamp')


In [23]:
faults_full.head()

,EquipmentID,EventTimeStamp,derate_2_6_hr,derate_6_12_hr,derate_12_24_hr,RecordID,eventDescription,spn,fmi,active,...,IgnStatus,IntakeManifoldTemperature,LampStatus,ParkingBrake,ServiceDistance,Speed,SwitchedBatteryVoltage,Throttle,TurboBoostPressure,derate_flag
1055070,2015,2000-03-18 19:14:10,0,0,0,1211418.0,High Voltage (Fuel Level),96.0,3.0,True,...,True,127.4,1279,False,NaN,0,NaN,100,0.58,0
1055069,2015,2000-03-18 19:14:10,0,0,0,1211417.0,High Voltage (Left Fuel Level Sensor),829.0,3.0,True,...,True,127.4,1279,False,NaN,0,NaN,100,0.58,0
1055067,2015,2000-03-18 19:20:47,0,0,0,1211419.0,High Voltage (Fuel Level),96.0,3.0,False,...,NaN,NaN,255,NaN,NaN,NaN,NaN,NaN,NaN,0
1055068,2015,2000-03-18 19:20:47,0,0,0,1211420.0,High Voltage (Left Fuel Level Sensor),829.0,3.0,False,...,NaN,NaN,255,NaN,NaN,NaN,NaN,NaN,NaN,0
1055066,1849,2000-03-19 02:59:58,0,0,0,1211422.0,Not Reporting Data Wheel Sensor ABS Axle 2 Right,792.0,7.0,False,...,NaN,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN,0


In [24]:
# Dropping columns we won't need for our model
faults_full = faults_full.drop(columns=['RecordID','derate_flag','Latitude','Longitude','LocationTimeStamp','eventDescription'])

Exploring missing data:

In [25]:
# Looking at nan proportions overall 
faults_full.isnull().mean().sort_values(ascending=False)

ServiceDistance              0.999813
SwitchedBatteryVoltage       0.899492
FuelTemperature              0.743542
ParkingBrake                 0.665183
Throttle                     0.644910
FuelLevel                    0.569229
AcceleratorPedal             0.545368
CruiseControlActive          0.507420
CruiseControlSetSpeed        0.506419
EngineTimeLtd                0.501480
TurboBoostPressure           0.499820
Speed                        0.499373
EngineOilTemperature         0.499287
FuelRate                     0.498547
FuelLtd                      0.498355
EngineLoad                   0.498292
DistanceLtd                  0.497971
EngineCoolantTemperature     0.497848
BarometricPressure           0.497837
IntakeManifoldTemperature    0.497748
EngineOilPressure            0.497736
EngineRpm                    0.497437
IgnStatus                    0.481077
EventTimeStamp               0.000000
activeTransitionCount        0.000000
active                       0.000000
LampStatus  

In [26]:
# Dropping service distance since it is mostly nulls
faults_full = faults_full.drop(columns=['ServiceDistance'])

In [27]:
# Investigating values that contain commas
comma_cols = [col for col in faults_full.columns if faults_full[col].astype(str).str.contains(',').any()]
for col in comma_cols:
    print(faults_full[faults_full[col].str.contains(',', na=False)][col].head())

977464     4,8
965593    51,2
958898    99,2
948149    99,6
936868    16,8
Name: AcceleratorPedal, dtype: object
1038706     14,355
1002871     14,355
977464      14,355
975979     14,4275
965593     14,2825
Name: BarometricPressure, dtype: object
1038706    113017,9
1002871    127046,6
977464     136467,8
975979     137217,6
965593     140927,4
Name: DistanceLtd, dtype: object
1038706    186,8
1002871    186,8
975979     183,2
965593     183,2
958898     183,2
Name: EngineCoolantTemperature, dtype: object
1038706    75,98
1002871    73,66
977464     80,62
975979     55,68
965593      66,7
Name: EngineOilPressure, dtype: object
1038706     177,575
1002871    180,1062
977464     189,6687
975979     197,2063
965593     187,3062
Name: EngineOilTemperature, dtype: object
1038706    1570,75
1002871     1725,5
965593     1391,75
955542     1623,25
936868      1414,5
Name: EngineRpm, dtype: object
1038706    2435,65
1002871    2744,05
977464      2956,9
975979      2973,9
965593     3056,35
N

Examining the output above, it appears that all the commas are meant to be decimal points (rather than legitimate commas separating hundreds and thousands, for example). 

In [28]:
# Replacing commas with decimal points
for col in comma_cols:
    faults_full[col] = faults_full[col].str.replace(',', '.', regex=False)

In [29]:
# Split EventTimeStamp column into year, month, day, and hour, then drop EventTimeStamp
faults_full['year'] = faults_full['EventTimeStamp'].dt.year
faults_full['month'] = faults_full['EventTimeStamp'].dt.month
faults_full['weekday'] = faults_full['EventTimeStamp'].dt.weekday
faults_full['hour'] = faults_full['EventTimeStamp'].dt.hour

# Also, we'll create a new delta column capturing the time since last fault.
faults_full['min_since_last_fault'] = faults_full.groupby('EquipmentID')['EventTimeStamp'].diff().dt.total_seconds() / 60

In [30]:
# Convert 'object' columns into numerical and string datatypes, as appropriate
numeric_features = ['min_since_last_fault','year','DistanceLtd','EngineTimeLtd','FuelLtd','activeTransitionCount','TurboBoostPressure','AcceleratorPedal','BarometricPressure','CruiseControlSetSpeed','EngineCoolantTemperature','EngineLoad','EngineOilPressure','EngineOilTemperature','EngineRpm','FuelLevel','FuelRate','FuelTemperature','IntakeManifoldTemperature','Speed','SwitchedBatteryVoltage','Throttle']
for feature in numeric_features:
    faults_full[feature] = faults_full[feature].astype('float64')

binary_features = ['active','CruiseControlActive','IgnStatus','ParkingBrake']
categorical_features = ['LampStatus','spn','fmi']
for feature in categorical_features:
    faults_full[feature] = faults_full[feature].astype('str')

In [31]:
# We'll apply a forward fill to deal with null values in any columns that represent a running count
ffill_features = ['DistanceLtd', 'EngineTimeLtd', 'FuelLtd']
for feature in ffill_features:
    faults_full[feature] = faults_full.groupby('EquipmentID')[feature].ffill().bfill()
    

In [32]:
# We'll use a random fill to deal with nulls in the min_since_last_fault column
faults_full['min_since_last_fault'] = faults_full.groupby('EquipmentID')['min_since_last_fault'].bfill().ffill()

# Get random samples equal to the number of missing values
random_samples = faults_full['min_since_last_fault'].dropna().sample(faults_full['min_since_last_fault'].isnull().sum(), random_state=0)

# Match indices of random samples with NaN indices
random_samples.index = faults_full[faults_full['min_since_last_fault'].isnull()].index

# Fill missing values with the randomly generated values
faults_full.loc[faults_full['min_since_last_fault'].isnull(), 'min_since_last_fault'] = random_samples


In [33]:
# For components like hours (0–23), we'll use sine and cosine transformations can help the model understand that hour 23 is close to hour 0
# Article here: https://feature-engine.trainindata.com/en/1.8.x/user_guide/creation/CyclicalFeatures.html

# Define transformation functions
def sin_transformer(period):
    return FunctionTransformer(lambda x: np.sin(x / period * 2 * np.pi), feature_names_out="one-to-one")

def cos_transformer(period):
    return FunctionTransformer(lambda x: np.cos(x / period * 2 * np.pi), feature_names_out="one-to-one")

# Numeric pipeline
numeric_transformer = Pipeline(steps=[
    # Applying an iterative imputer to fill missing values
    ("imputer", IterativeImputer(random_state=0, verbose=2)),
    # Applying a MaxAbsScaler (better than StandardScaler for sparse inputs or inputs that are not normally distributed)
    ("scaler", MaxAbsScaler())
])

# Categorical pipeline
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Cyclical pipeline for features that repeat in a cycle (hours, days, and months)
cyclical_transformer = ColumnTransformer(transformers=[
    ("month_sin", sin_transformer(12), ["month"]),
    ("month_cos", cos_transformer(12), ["month"]),
    ("weekday_sin", sin_transformer(7), ["weekday"]),
    ("weekday_cos", cos_transformer(7), ["weekday"]),
    ("hour_sin", sin_transformer(24), ["hour"]),
    ("hour_cos", cos_transformer(24), ["hour"]),
])

# Combine all the transformations into 1 preprocessor
# Using ColumnTransformer to select specific columns and apply transformations
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features+binary_features),
        ("cycle", cyclical_transformer, ["month", "weekday", "hour"])
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

pipe = Pipeline(
    steps=[("preprocessor", preprocessor)]
)

In [34]:
# Anything before 1/1/2019 is our training/validation data
faults_train = faults_full[~(faults_full['EventTimeStamp'].dt.date >= pd.to_datetime('2019-01-01').date())]

# Converting EventTimeStamp to datetime
faults_train['EventTimeStamp'] = pd.to_datetime(faults_train['EventTimeStamp'])

# Makse sure dataframe is sorted by timestamp
faults_train = faults_train.sort_values(by='EventTimeStamp')

# Dropping EquipmentID and EventTimeStamp since they won't be used in the model
faults_train = faults_train.drop(columns=['EquipmentID','EventTimeStamp'])

/var/folders/51/zgq0lbb14t13h0_bgn8nntnm0000gn/T/ipykernel_91283/3895464975.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  faults_train['EventTimeStamp'] = pd.to_datetime(faults_train['EventTimeStamp'])


Here, we'll split our data into 5 equal parts and apply the pre-processor pipeline (scaling and filling in missing values).

In [35]:
# Define our model features 
features = numeric_features + categorical_features + binary_features + ['month','weekday','hour']

# Create 5 equal-sized arrays from faults_train
parts = np.array_split(faults_train, 5)

# Initialize an empty list to keep track of the training data
train_parts = []

for i in range(0, 4):
    # Add the current iteration number to the training list
    train_parts.append(parts[i])

    # Concatenate the training datasets 
    train_set = pd.concat(train_parts)
    
    # The validation set is always the next number in the sequence
    val_set = parts[i + 1]

    # Fit and transform the train set
    train_clean = pipe.fit_transform(train_set)
    
    # Transform the validation set 
    val_clean = pipe.transform(val_set)

    # Attach feature names
    feature_names = pipe.get_feature_names_out()
    
    # Save the cleaned data to csv
    # As an alternative to csv, we could also use pickle format or use duckdb (https://duckdb.org/docs/current/guides/python/jupyter)
    train_clean_df = pd.DataFrame(train_clean)
    train_clean_df.columns = feature_names
    train_clean_df.to_csv(f'../data/train_set_{i+1}.csv')
    
    val_clean_df = pd.DataFrame(val_clean)
    val_clean_df.columns = feature_names
    val_clean_df.to_csv(f'../data/val_set_{i+1}.csv')
  

/opt/anaconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


[IterativeImputer] Completing matrix with shape (188771, 22)
[IterativeImputer] Ending imputation round 1/10, elapsed time 1.57
[IterativeImputer] Change: 12647.544505339149, scaled tolerance: 9371.996783333334 
[IterativeImputer] Ending imputation round 2/10, elapsed time 3.10
[IterativeImputer] Change: 292.88280141192234, scaled tolerance: 9371.996783333334 
[IterativeImputer] Early stopping criterion reached.
[IterativeImputer] Completing matrix with shape (188771, 22)
[IterativeImputer] Ending imputation round 1/2, elapsed time 0.21
[IterativeImputer] Ending imputation round 2/2, elapsed time 0.41
[IterativeImputer] Completing matrix with shape (377542, 22)
[IterativeImputer] Ending imputation round 1/10, elapsed time 3.08
[IterativeImputer] Change: 16707.913177887407, scaled tolerance: 9371.996783333334 
[IterativeImputer] Ending imputation round 2/10, elapsed time 6.36
[IterativeImputer] Change: 1775.565447267068, scaled tolerance: 9371.996783333334 
[IterativeImputer] Early stop

In [36]:
# Define a way to evaluate the amount of money saved by the model
# This will help us compare different models better 
def calculate_cost_savings(y_true, y_pred):
    """
    Returns the amount of money we are saving Big G by implementing our model
    """
    # ravel() extracts values in order: TN, FP, FN, TP
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    # Calculate total cost
    cost = (tp * 4000) - (fp * 500) - (fn * 4000) 
    
    return cost

Now that we have our pre-processed cross-validation datasets, we'll fit a model to each of the sets and evaluate its effectiveness. We'll apply 5-fold cross-validation to evaluate how well our model does. 
Good article here: https://www.geeksforgeeks.org/machine-learning/k-fold-cross-validation-in-machine-learning/

Using scale_pos_weight to address the class imbalance:

In [ ]:
# Define our model features 
features = numeric_features + categorical_features + binary_features + ['month','weekday','hour']

# For now we will run separate models for each target time window
# To evaluate multiple targets in the same model, we could look into a multiclass predictor
# To evaluate, then we'd use predict_proba (instead of using .predict)

targets = ['derate_2_6_hr','derate_6_12_hr','derate_12_24_hr']

for target in targets:
    print(f'TARGET: {target}')
    
    # Initialize empty lists to keep track of the model evaluation metrics
    savings = []
    metrics = []
    
    for i in range(1, 5):
        # Read in pre-processed train and validation data
        train_set = pd.read_csv(f'../data/train_set_{i}.csv')
        val_set = pd.read_csv(f'../data/val_set_{i}.csv')
    
        # Split into x (features) and y (target)
        X_train = train_set[[col for col in train_set.columns if not col.startswith('derate')]]
        y_train = train_set[target]
    
        X_val = val_set[[col for col in train_set.columns if not col.startswith('derate')]]
        y_val = val_set[target]

        # Defining the scale_pos_weight hyperparameter
        # Article here: https://machinelearningmastery.com/xgboost-for-imbalanced-classification/
        if (train_set[target].value_counts().loc[1.0] == 0):
            param = train_set[target].value_counts().loc[0.0]
        else:
            param = train_set[target].value_counts().loc[0.0]/train_set[target].value_counts().loc[1.0]
        
        # Initialize and fit a model
        model = XGBClassifier(n_estimators=1000, scale_pos_weight=param).fit(X_train, y_train)
        
        # Generate predictions on the validation set using the model
        y_pred = model.predict(X_val)
    
        # Calculate the cost savings associated with the model's output
        cost_savings = calculate_cost_savings(y_val, y_pred)
        savings.append(cost_savings)
    
        print(f'Cost savings for fold #{i}: ${cost_savings}')

        # Print confusion matrix and classification report
        print(confusion_matrix(y_val, y_pred))
        print(classification_report(y_val, y_pred))

        # Calculate f-beta score (beta=8 since a false negative is 8x more costly than a false positive)
        fbeta = fbeta_score(y_val, y_pred, beta=8)
        metrics.append(fbeta)

        print(f'F-beta score for fold #{i}: {fbeta}')

    
    print(f'Average cost savings for this model: ${pd.Series(savings).mean()}') 
    print(f'Average f-beta score for this model: ${pd.Series(metrics).mean()}') 
    print()

Using SMOTE to address the class imbalance:

In [ ]:
# Define our model features 
features = numeric_features + categorical_features + binary_features + ['month','weekday','hour']

# For now we will run separate models for each target time window
# To evaluate multiple targets in the same model, we could look into a multiclass predictor
# To evaluate, then we'd use predict_proba (instead of using .predict)

targets = ['derate_2_6_hr','derate_6_12_hr','derate_12_24_hr']

for target in targets:
    print(f'TARGET: {target}')
    
    # Initialize empty lists to keep track of the model evaluation metrics
    savings = []
    metrics = []
    
    for i in range(1, 5):
        # Read in pre-processed train and validation data
        train_set = pd.read_csv(f'../data/train_set_{i}.csv')
        val_set = pd.read_csv(f'../data/val_set_{i}.csv')
    
        # Split into x (features) and y (target)
        X_train = train_set[[col for col in train_set.columns if not col.startswith('derate')]]
        y_train = train_set[target]
    
        X_val = val_set[[col for col in train_set.columns if not col.startswith('derate')]]
        y_val = val_set[target]

        # Apply SMOTE oversampling to deal with imbalanced classes
        smote = SMOTE(random_state=321)

        X_train_smote, y_train_smote = smote.fit_resample(
            X_train,
            y_train
        )
        
        # Initialize and fit a model
        model = XGBClassifier(n_estimators=1000).fit(X_train_smote, y_train_smote)
        
        # Generate predictions on the validation set using the model
        y_pred = model.predict(X_val)
    
        # Calculate the cost savings associated with the model's output
        cost_savings = calculate_cost_savings(y_val, y_pred)
        savings.append(cost_savings)
    
        print(f'Cost savings for fold #{i}: ${cost_savings}')

        # Print confusion matrix and classification report
        print(confusion_matrix(y_test, y_pred))
        print(classification_report(y_test, y_pred))

        # Calculate f-beta score (beta=8 since a false negative is 8x more costly than a false positive)
        fbeta = fbeta_score(y_test, y_pred, beta=8)
        metrics.append(fbeta)

        print(f'F-beta score for fold #{i}: ${fbeta}')

    
    print(f'Average cost savings for this model: ${pd.Series(savings).mean()}') 
    print(f'Average f-beta score for this model: ${pd.Series(metrics).mean()}') 
    print()

Seeing how well the model does on our "real-world" data:

In [ ]:
# Anything before 1/1/2019 is our training data
train_set = faults_full[~(faults_full['EventTimeStamp'].dt.date >= pd.to_datetime('2019-01-01').date())]
# Dropping EquipmentID and EventTimeStamp since they won't be used in the model
train_set = train_set.drop(columns=['EquipmentID','EventTimeStamp'])

# Data on or after 1/1/2019 is our final "real-world" test dataset
test_set = faults_full[faults_full['EventTimeStamp'].dt.date >= pd.to_datetime('2019-01-01').date()]
# Dropping EquipmentID and EventTimeStamp since they won't be used in the model
test_set = test_set.drop(columns=['EquipmentID','EventTimeStamp'])

# Fit and transform the train set using our preprocessing pipeline
train_clean = pipe.fit_transform(train_set)

# Transform the test set
test_clean = pipe.transform(test_set)

# Get feature names from preprocessing pipeline
feature_names = pipe.get_feature_names_out()

# Attach feature names
train_clean_df = pd.DataFrame(train_clean)
train_clean_df.columns = feature_names

test_clean_df = pd.DataFrame(test_clean)
test_clean_df.columns = feature_names

# Split into x (features) and y (target)
X_train = train_clean_df[[col for col in train_clean_df.columns if not col.startswith('derate')]]
y_train = train_clean_df['derate_6_12_hr']

X_test = test_clean_df[[col for col in test_clean_df.columns if not col.startswith('derate')]]
y_test = test_clean_df['derate_6_12_hr']

# Defining the scale_pos_weight hyperparameter
# Article here: https://machinelearningmastery.com/xgboost-for-imbalanced-classification/
param = train_set['derate_6_12_hr'].value_counts().loc[0.0]/train_set['derate_6_12_hr'].value_counts().loc[1.0]

# Initialize and fit a model
model = XGBClassifier(n_estimators=1000, scale_pos_weight=param).fit(X_train, y_train)

# Generate predictions on the test set using the model
y_pred = model.predict(X_test)

# Calculate the cost savings associated with the model's output
cost_savings = calculate_cost_savings(y_test, y_pred)

print(f'Cost savings on real-world data: ${cost_savings}.')

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Calculate f-beta score (beta=8 since a false negative is 8x more costly than a false positive)
fbeta = fbeta_score(y_test, y_pred, beta=8)

print(f'F-beta score for real-world data: ${fbeta}')

In [ ]:
pd.DataFrame({
    'variable': X_test.columns,
    'importance': permutation_importance(model, X_test, y_test, random_state = 321)['importances_mean']
}).sort_values('importance', ascending = False)